# 🗂️ Subset & Model Selection
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, which ones should you include in the model? Subset selection methods systematically compare model subsets. Information criteria (Cp, BIC, adjusted R²) give principled ways to choose model size without a test set.

### What you'll learn
- Best subset selection: exhaustive search
- Forward and backward stepwise selection: greedy approximations
- Cp, BIC, and adjusted R² as model selection criteria
- Practical comparison with cross-validation

### Dataset: Hitters (ISLP) — 19 baseball predictors
### Time: ~50 minutes

## 🎯 Part 1 — Why Variable Selection?

With p predictors, using all of them can overfit — especially when p is large relative to n. Including irrelevant predictors:
- Increases variance of estimates
- Reduces interpretability
- May worsen prediction on new data

We want the smallest model that explains the data well.

In [ ]:
# Forward stepwise selection (greedy — each step adds best remaining feature)
def forward_stepwise(X, y, feature_names, max_features=None):
    if max_features is None: max_features = X.shape[1]
    selected = []
    remaining = list(range(X.shape[1]))
    results = []
    lr = LinearRegression()
    for step in range(max_features):
        best_score, best_feat = -np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            cv = cross_val_score(lr, X[:, candidate], y, cv=5,
                                scoring='neg_mean_squared_error').mean()
            if cv > best_score:
                best_score, best_feat = cv, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        results.append({'n_features': step+1, 'features': selected.copy(),
                       'cv_mse': -best_score,
                       'last_added': feature_names[best_feat]})
    return pd.DataFrame(results)

print('Running forward stepwise selection...')
fwd_results = forward_stepwise(X, y, features, max_features=15)
print(fwd_results[['n_features','last_added','cv_mse']].to_string())
print(f'\nBest model size: {fwd_results["cv_mse"].idxmin()+1} features')

In [ ]:
# Plot: CV MSE vs number of features
fig, ax = plt.subplots(figsize=(10, 4))
best_n = fwd_results['cv_mse'].idxmin()
ax.plot(fwd_results['n_features'], fwd_results['cv_mse'], 'o-', color='#1e5fa8', lw=2, markersize=6)
ax.axvline(fwd_results.loc[best_n, 'n_features'], color='#e85d20', lw=1.5, ls='--',
          label=f'Best: {fwd_results.loc[best_n,"n_features"]} features (CV MSE={fwd_results.loc[best_n,"cv_mse"]:.4f})')
ax.set_xlabel('Number of features')
ax.set_ylabel('5-fold CV MSE (log salary)')
ax.set_title('Forward Stepwise Selection — Hitters Dataset')
ax.legend()
plt.tight_layout(); plt.show()
best_features = fwd_results.loc[best_n, 'features']
print(f'\nSelected features: {[features[i] for i in best_features]}')

In [ ]:
# Information criteria: AIC, BIC, adjusted R²
# (computed on full training data — no cross-validation needed)
import statsmodels.api as sm

n = len(y)
X_sm = sm.add_constant(X[:, :10])  # first 10 features for illustration
model_sm = sm.OLS(y, X_sm).fit()

print('Model selection criteria (statsmodels):')  
print(f'  AIC:  {model_sm.aic:.2f}')
print(f'  BIC:  {model_sm.bic:.2f}')
print(f'  R²:   {model_sm.rsquared:.4f}')
print(f'  Adj R²: {model_sm.rsquared_adj:.4f}')
print()
print('Building models of increasing size and comparing criteria...')
for k in [1, 3, 5, 8, 10, 12, 15]:
    if k <= len(features):
        Xk = sm.add_constant(X[:, :k])
        mk = sm.OLS(y, Xk).fit()
        print(f'  p={k:<3} AIC={mk.aic:.1f}  BIC={mk.bic:.1f}  AdjR²={mk.rsquared_adj:.4f}')

In [ ]:
# Exercise workspace
# Task 1: Implement backward stepwise selection (start with all features, remove worst one each step)
# YOUR CODE HERE

# Task 2: Compare forward stepwise vs Lasso (from regularization notebook) on Hitters
# Which selects fewer features? Which has lower CV MSE?
from sklearn.linear_model import LassoCV
# YOUR CODE HERE

# Task 3: What features does the best 5-feature model select? Interpret each one
best_5 = fwd_results.loc[fwd_results['n_features']==5, 'features'].values[0]
print('Top 5 features:', [features[i] for i in best_5])
# YOUR CODE HERE — fit and report coefficients

In [ ]:
# @title 📝 Quiz — Subset & Model Selection
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is best subset selection and why is it computationally expensive?
# @markdown **Q2:** What is the difference between forward and backward stepwise selection?
# @markdown **Q3:** What does BIC penalize compared to AIC?
# @markdown **Q4:** Why is adjusted R² better than R² for model comparison?
# @markdown **Q5:** When would you use stepwise selection vs Lasso for variable selection?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Subset & Model Selection"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
